## **PROJET TINDER - EDA**

### **5. VISUALISATIONS FINALES & STORYTELLING**

- Input  : `outputs/data/df_speed_dating_cleaned.csv` 
- Livrable : visualisations prêtes pour une présentation équipe marketing

In [27]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path
 
pio.templates.default = "plotly_white"
 
# ── Palette ──────────────────────────────────────────────────────────────────
C = {
    "match":    "#1D9E75",
    "no_match": "#E24B4A",
    "male":     "#378ADD",
    "female":   "#D4537E",
    "purple":   "#7F77DD",
    "amber":    "#EF9F27",
    "teal":     "#0FA98E",
    "coral":    "#D85A30",
    "neutral":  "#888780",
    "dark":     "#1A1A2E",
    "light_bg": "#F8F7F4",
}
 
ATTR6    = ["attr",  "sinc",  "intel",  "fun",   "amb",   "shar"]
ATTR6_FR = ["Attractivité","Sincérité","Intelligence","Fun","Ambition","Intérêts communs"]
PREF_COLS = ["attr1_1","sinc1_1","intel1_1","fun1_1","amb1_1","shar1_1"]
SCORE_OTHER = ["attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o","like_o","prob_o"]
SCORE_SELF  = ["attr","sinc","intel","fun","amb","shar","like","prob"]

PROJECT_ROOT = Path('../')
INPUT_DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'df_speed_dating_prep.csv'
print(INPUT_DATA_PATH)

../outputs/data/df_speed_dating_prep.csv


In [28]:
# =============================================================================
# CHARGEMENT + PREP
# =============================================================================
df = pd.read_csv(INPUT_DATA_PATH)
print(f"\nDataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
 
df["age_diff"] = (df["age"] - df["age_o"]).abs()
df["score_received_avg"] = df[SCORE_OTHER].mean(axis=1).round(2)
df["attr_expectation_gap"] = (df["attr1_1"] - df["attr_o"]).round(2)
df["gender_label"] = df["gender"].map({0:"Femme", 1:"Homme"})
df["match_label"] = df["match"].map({0:"Pas de match", 1:"Match"})
 
# ── Prétraitements ───────
match_rate = df["match"].mean() * 100
dec_f = df[df["gender"]==0]["dec"].mean() * 100
dec_m = df[df["gender"]==1]["dec"].mean() * 100
 
deltas, corrs = {}, {}
for col in ATTR6:
    m0 = df[df["match"]==0][col].mean()
    m1 = df[df["match"]==1][col].mean()
    deltas[col] = round(m1 - m0, 3)
    corrs[col]  = round(df[col].corr(df["match"]), 3)
 
like_thresholds = [4, 5, 6, 7, 8, 9]
like_rates = [df[df["like"]>=t]["match"].mean()*100 for t in like_thresholds]
like_ratio = df[df["like"]>=8]["match"].mean() / df["match"].mean()
 
gender_deltas = {}
for col in ATTR6[:5]:
    df_f = df[df["gender"]==0]
    df_m = df[df["gender"]==1]
    gender_deltas[col] = (
        df_f[df_f["match"]==1][col].mean() - df_f[df_f["match"]==0][col].mean(),
        df_m[df_m["match"]==1][col].mean() - df_m[df_m["match"]==0][col].mean(),
    )
 
# Pref T1 vs réel
decl_f, reel_f, decl_m, reel_m = [], [], [], []
for p, s in zip(PREF_COLS[:5], ATTR6[:5]):
    decl_f.append(df[df["gender"]==0][p].mean())
    reel_f.append(df[df["gender"]==0][s].mean())
    decl_m.append(df[df["gender"]==1][p].mean())
    reel_m.append(df[df["gender"]==1][s].mean())
 
print("Prétraitements terminés")


Dataset : 8,378 lignes × 48 colonnes
Prétraitements terminés


In [29]:
# =============================================================================
# DASHBOARD EXÉCUTIF
# =============================================================================
print("\n" + "─" * 65)
print("DASHBOARD EXÉCUTIF")
print("─" * 65)
 
 
best_attr   = max(deltas, key=deltas.get)
best_delta  = deltas[best_attr]
max_t1_corr = max(abs(df[p].corr(df["match"])) for p in PREF_COLS)
like8_rate  = df[df["like"]>=8]["match"].mean() * 100
 
# 4 KPI indicators
fig_v1 = go.Figure()
kpis = [
    {"mode":"gauge+number","value":match_rate,
     "title":{"text":"Taux match global","font":{"size":13}},
     "number":{"suffix":"%","font":{"size":32,"color":C["match"]}},
     "gauge":{"axis":{"range":[0,50]},"bar":{"color":C["match"]}},
     "domain":{"x":[0.0,0.24],"y":[0.0,1.0]}},
    {"mode":"number","value":best_delta,
     "title":{"text":f"Différence {ATTR6_FR[ATTR6.index(best_attr)]}<br>(meilleur attribut)","font":{"size":13}},
     "number":{"prefix":"+","valueformat":".2f","font":{"size":36,"color":C["amber"]}},
     "domain":{"x":[0.26,0.50],"y":[0.0,1.0]}},
    {"mode":"number","value":max_t1_corr,
     "title":{"text":"Max corr. préf. T1 vs match<br>(déclarer ≠ prédire)","font":{"size":13}},
     "number":{"suffix":"~ 0","valueformat":".3f","font":{"size":36,"color":C["no_match"]}},
     "domain":{"x":[0.52,0.74],"y":[0.0,1.0]}},
    {"mode":"number","value":like8_rate,
     "title":{"text":"Match rate si like≥8<br>(2.1x la base)","font":{"size":13}},
     "number":{"suffix":"%","valueformat":".1f","font":{"size":36,"color":C["purple"]}},
     "domain":{"x":[0.76,1.0],"y":[0.0,1.0]}},
]
for kpi in kpis:
    fig_v1.add_trace(go.Indicator(**kpi))
fig_v1.update_layout(
    title={"text":"Dashboard exécutif : 4 KPIs clés",
           "font":{"size":18,"color":C["dark"]},"x":0.5,"xanchor":"center"},
    height=280, paper_bgcolor=C["light_bg"],
)
fig_v1.show()
 
# Barre delta attributs
attrs_sorted = sorted(ATTR6, key=lambda c: deltas[c], reverse=True)
delta_colors_v1 = [C["match"] if deltas[c]>=1.3 else C["amber"] if deltas[c]>=0.9
                   else C["neutral"] for c in attrs_sorted]

fig_v1b = go.Figure(go.Bar(
    x=[ATTR6_FR[ATTR6.index(c)] for c in attrs_sorted],
    y=[deltas[c] for c in attrs_sorted],
    marker_color=delta_colors_v1,
    text=[f"+{deltas[c]:.2f}" for c in attrs_sorted],
    textposition="outside",
))
fig_v1b.add_hline(y=1.0, line_dash="dot", line_color=C["match"],
    annotation_text="Signal fort")
fig_v1b.update_layout(
    title={"text":"Force prédictive des attributs (match=1 vs match=0)",
           "font":{"size":15,"color":C["dark"]},"x":0.5,"xanchor":"center"},
    yaxis_title="Delta score (0-10)", height=360,
    plot_bgcolor=C["light_bg"], paper_bgcolor=C["light_bg"], showlegend=False,
)
fig_v1b.show()


─────────────────────────────────────────────────────────────────
DASHBOARD EXÉCUTIF
─────────────────────────────────────────────────────────────────


In [30]:
# =============================================================================
# PODIUM DES ATTRIBUTS
# =============================================================================
print("\n" + "─" * 65)
print("PODIUM DES ATTRIBUTS")
print("─" * 65)
 
# Tri par delta décroissant
sorted_pairs = sorted(zip(ATTR6_FR, ATTR6), key=lambda x: deltas[x[1]], reverse=True)
labels_sorted  = [fr for fr, _ in sorted_pairs]
deltas_sorted  = [deltas[col] for _, col in sorted_pairs]
corrs_sorted   = [corrs[col]  for _, col in sorted_pairs]
 
bar_colors = []
for d in deltas_sorted:
    if d >= 1.3:   bar_colors.append(C["match"])
    elif d >= 0.9: bar_colors.append(C["amber"])
    else:          bar_colors.append(C["neutral"])
 
fig_v2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Match rate moyen",
        "Corrélation avec match",
    ],
    horizontal_spacing=0.12,
)
 
# Delta bars
fig_v2.add_trace(go.Bar(
    y=labels_sorted[::-1],
    x=deltas_sorted[::-1],
    orientation="h",
    marker_color=bar_colors[::-1],
    marker_line_width=0,
    text=[f"+{d:.3f}" for d in deltas_sorted[::-1]],
    textposition="outside",
    textfont=dict(size=13, color=C["dark"]),
    showlegend=False,
), row=1, col=1)
fig_v2.add_vline(x=1.0, line_dash="dot", line_color=C["match"],
    annotation_text="Seuil fort", row=1, col=1)
fig_v2.add_vline(x=0.7, line_dash="dot", line_color=C["amber"],
    annotation_text="Seuil modéré", row=1, col=1)
 
# Corrélation dots
fig_v2.add_trace(go.Scatter(
    y=labels_sorted[::-1],
    x=corrs_sorted[::-1],
    mode="markers+text",
    marker=dict(
        color=bar_colors[::-1],
        size=[40 if d >= 1.3 else 28 if d >= 0.9 else 20
              for d in deltas_sorted[::-1]],
        symbol="circle",
        line=dict(width=2, color="white"),
    ),
    text=[f"{c:+.3f}" for c in corrs_sorted[::-1]],
    textposition="middle center",
    textfont=dict(size=11, color="white", family="monospace"),
    showlegend=False,
), row=1, col=2)

fig_v2.add_vline(x=0.2, line_dash="dot", line_color=C["match"],
    annotation_text="Signal fort", row=1, col=2)
 
fig_v2.update_layout(
    title={
        "text": "Classement des attributs : Fun & Attractivité dominent, "
                "Sincérité & Intelligence sont des signaux faibles",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=420, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
)
fig_v2.update_xaxes(title_text="score (0-10)", row=1, col=1)
fig_v2.update_xaxes(title_text="Corrélation de Pearson", row=1, col=2)
fig_v2.show()


─────────────────────────────────────────────────────────────────
PODIUM DES ATTRIBUTS
─────────────────────────────────────────────────────────────────


In [31]:
# =============================================================================
# CE QU'ON DIT VS CE QU'ON FAIT
# =============================================================================
print("\n" + "─" * 65)
print("CE QU'ON DIT VS CE QU'ON FAIT")
print("─" * 65)
 
attrs5_fr = ["Attractivité","Sincérité","Intelligence","Fun","Ambition"]
 
fig_v3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Femmes — déclaré T1 vs notes réelles soirée",
        "Hommes — déclaré T1 vs notes réelles soirée",
    ],
    horizontal_spacing=0.14,
)
 
for col_idx, (gval, d_vals, r_vals, clr) in enumerate([
    (0, decl_f, reel_f, C["female"]),
    (1, decl_m, reel_m, C["male"]),
]):
    fig_v3.add_trace(go.Bar(
        x=attrs5_fr, y=d_vals,
        name="Déclaré avant (T1)",
        marker_color=clr,
        marker_pattern_shape="/",
        opacity=0.55,
        showlegend=(col_idx == 0),
        legendgroup="declared",
    ), row=1, col=col_idx+1)
    fig_v3.add_trace(go.Bar(
        x=attrs5_fr, y=r_vals,
        name="Noté réellement (soirée)",
        marker_color=clr,
        opacity=0.95,
        showlegend=(col_idx == 0),
        legendgroup="real",
    ), row=1, col=col_idx+1)
 
fig_v3.update_layout(
    title={
        "text": "Ce qu'on dit vouloir vs ce qu'on fait\n",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=440, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
    barmode="group", legend_title="Type de mesure",
)

fig_v3.update_yaxes(title_text="Score moyen (0-10)")
fig_v3.add_annotation(
    text="Les scores 'réels' sont bien plus hauts car les gens notent généreusement "
         "en face à face — mais c'est la VARIANCE qui prédit le match, pas le niveau absolu.",
    xref="paper", yref="paper", x=0.5, y=-0.18,
    showarrow=False, font=dict(size=11, color=C["neutral"]),
    align="center",
)
fig_v3.show()


─────────────────────────────────────────────────────────────────
CE QU'ON DIT VS CE QU'ON FAIT
─────────────────────────────────────────────────────────────────


In [32]:
# =============================================================================
# LE SUPERPRÉDICTEUR LIKE
# =============================================================================
print("\n" + "─" * 65)
print("LE SUPERPRÉDICTEUR LIKE")
print("─" * 65)
 
fig_v4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Taux de match selon le score 'like' minimum",
        "Distribution du score like par résultat",
    ],
    horizontal_spacing=0.12,
)
 
# Courbe taux match
fig_v4.add_trace(go.Scatter(
    x=like_thresholds, y=like_rates,
    mode="lines+markers",
    name="Taux de match",
    marker=dict(color=C["match"], size=12,
                line=dict(width=2, color="white")),
    line=dict(color=C["match"], width=3),
    fill="tozeroy", fillcolor="rgba(29,158,117,0.12)",
    showlegend=False,
), row=1, col=1)
 
fig_v4.add_hline(
    y=match_rate, line_dash="dash", line_color=C["neutral"],
    line_width=2,
    annotation_text=f"Base : {match_rate:.1f}%",
    annotation_position="bottom right",
    row=1, col=1,
)
 
# Zone d'action Tinder
fig_v4.add_vrect(
    x0=7.5, x1=9.5,
    fillcolor="rgba(29,158,117,0.1)",
    line_width=0,
    annotation_text="Zone Tinder",
    annotation_position="top left",
    row=1, col=1,
)
 
# Annotations clés
for t, r in zip(like_thresholds, like_rates):
    fig_v4.add_annotation(
        x=t, y=r + 1.2,
        text=f"{r:.1f}%",
        showarrow=False,
        font=dict(size=11, color=C["dark"], family="monospace"),
        row=1, col=1,
    )
 
# Violin like par match
for mv, clr, nm in [(0, C["no_match"], "Pas de match"), (1, C["match"], "Match")]:
    fig_v4.add_trace(go.Violin(
        y=df[df["match"]==mv]["like"],
        name=nm,
        marker_color=clr,
        box_visible=True, meanline_visible=True,
        opacity=0.82, width=0.7,
    ), row=1, col=2)
 
fig_v4.update_layout(
    title={
        "text": f"like ≥ 8 => {like8_rate:.0f}% de chance de match "
                f"({like_ratio:.1f}x la moyenne) — Le superprédicteur du dataset",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=440, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
    legend_title="Résultat",
)
fig_v4.update_xaxes(title_text="Seuil minimum de like", row=1, col=1,
    tickmode="array", tickvals=like_thresholds)
fig_v4.update_yaxes(title_text="Taux de match (%)", row=1, col=1)
fig_v4.update_yaxes(title_text="Score like (0-10)", row=1, col=2)
fig_v4.show()


─────────────────────────────────────────────────────────────────
LE SUPERPRÉDICTEUR LIKE
─────────────────────────────────────────────────────────────────


In [33]:
# =============================================================================
# RADAR HOMME VS FEMME
# =============================================================================
print("\n" + "─" * 65)
print("RADAR HOMME VS FEMME")
print("─" * 65)
 
attrs_radar = ATTR6_FR[:5]
delta_f_vals = [gender_deltas[c][0] for c in ATTR6[:5]]
delta_m_vals = [gender_deltas[c][1] for c in ATTR6[:5]]
 
fig_v5 = make_subplots(
    rows=1, cols=2,
    specs=[[{"type":"polar"}, {"type":"polar"}]],
    subplot_titles=[
        "Femme",
        "Homme",
    ],
)
 
for col_idx, (vals, clr, genre) in enumerate([
    (delta_f_vals, C["female"], "Femme"),
    (delta_m_vals, C["male"],   "Homme"),
]):
    fig_v5.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=attrs_radar + [attrs_radar[0]],
        fill="toself",
        name=genre,
        line_color=clr,
        fillcolor="rgba(55,138,221,0.20)" if clr == C["male"] else "rgba(212,83,126,0.20)",
        marker=dict(size=8, color=clr),
    ), row=1, col=col_idx+1)
 
fig_v5.update_layout(
    title={
        "text": "Sensibilité: Femmes plus sensibles au Fun & Attractivité",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=480,
    paper_bgcolor=C["light_bg"],
    polar=dict(radialaxis=dict(range=[0, 2], tickfont=dict(size=9))),
    polar2=dict(radialaxis=dict(range=[0, 2], tickfont=dict(size=9))),
    showlegend=False,
)
fig_v5.show()


─────────────────────────────────────────────────────────────────
RADAR HOMME VS FEMME
─────────────────────────────────────────────────────────────────


In [34]:
# =============================================================================
# MYTHES VS RÉALITÉ (ce qui ne prédit PAS le match)
# =============================================================================
print("\n" + "─" * 65)
print("MYTHES VS RÉALITÉ")
print("─" * 65)
 
mythe_vars  = ["Même race", "Intérêts communs", "Écart d'âge", "Ce qu'on déclare vouloir"]
mythe_corrs = [
    df["samerace"].corr(df["match"]),
    df["int_corr"].corr(df["match"]),
    -df["age_diff"].corr(df["match"]),
    max(abs(df[p].corr(df["match"])) for p in PREF_COLS),
]
reel_vars  = ["Fun", "Attractivité", "Intérêts communs (réel)", "Appréciation globale (like)"]
reel_corrs = [corrs["fun"], corrs["attr"], corrs["shar"],
              df["like"].corr(df["match"])]
 
fig_v6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Ce qui ne prédit pas le match (corrélation ~ 0)",
        "Ce qui prédit vraiment le match",
    ],
    horizontal_spacing=0.14,
)
 
fig_v6.add_trace(go.Bar(
    x=mythe_vars, y=[abs(c) for c in mythe_corrs],
    marker_color=C["no_match"], opacity=0.75,
    text=[f"{abs(c):.3f}" for c in mythe_corrs],
    textposition="outside", showlegend=False,
), row=1, col=1)
 
fig_v6.add_trace(go.Bar(
    x=reel_vars, y=reel_corrs,
    marker_color=C["match"], opacity=0.85,
    text=[f"+{c:.3f}" for c in reel_corrs],
    textposition="outside", showlegend=False,
), row=1, col=2)
 
fig_v6.add_hline(y=0.05, line_dash="dot", line_color=C["neutral"],
    annotation_text="Seuil négligeable", row=1, col=1)
 
fig_v6.update_layout(
    title={
        "text": "Mythes vs Réalité : la race, les intérêts communs et "
                "les préférences déclarées ne prédisent pas le match",
        "font": {"size": 16, "color": C["dark"]},
        "x": 0.5, "xanchor": "center",
    },
    height=420, plot_bgcolor=C["light_bg"],
    paper_bgcolor=C["light_bg"],
)
fig_v6.update_yaxes(title_text="Corrélation avec match (valeur absolue)")
fig_v6.show()


─────────────────────────────────────────────────────────────────
MYTHES VS RÉALITÉ
─────────────────────────────────────────────────────────────────


In [35]:
# =============================================================================
# SYNTHÈSE CONSOLE — STORYTELLING FINAL
# =============================================================================
print(f"""
{'=' * 65}
STORYTELLING FINAL TERMINÉ
{'=' * 65}

Sur 8 378 interactions de speed dating à Columbia, seulement 16.5% débouchent sur un match mutuel.
Ce qui crée un match ? Pas ce que les gens croient.
 
- La race en commun : +1.7pp seulement. Négligeable.
- Les intérêts communs : corr=0.031. Quasi nul.
- Ce qu'on déclare vouloir : corr max={max(abs(df[p].corr(df['match'])) for p in PREF_COLS):.3f} ≈ 0.

Les gens ne savent pas ce qui les attire.
 
- Ce qui marche vraiment :
    - Fun          ->  Delta=+1.41 si match
    - Attractivité  ->  Delta=+1.34 si match
    - Intérêts réels  ->  Delta=+1.37 (noté EN FACE, pas déclaré)
    - Like ≥ 8     ->  {like8_rate:.0f}% de match ({like_ratio:.1f}x la base)
 
 Recommandation #1 : Pondérer Fun & Attractivité dans l'algo.
 Recommandation #2 : Créer un 'like score' composite.
 Recommandation #3 : Simplifier l'onboarding — la race et les intérêts déclarés n'ont pas de valeur prédictive."
""")


STORYTELLING FINAL TERMINÉ

Sur 8 378 interactions de speed dating à Columbia, seulement 16.5% débouchent sur un match mutuel.
Ce qui crée un match ? Pas ce que les gens croient.

- La race en commun : +1.7pp seulement. Négligeable.
- Les intérêts communs : corr=0.031. Quasi nul.
- Ce qu'on déclare vouloir : corr max=0.015 ≈ 0.

Les gens ne savent pas ce qui les attire.

- Ce qui marche vraiment :
    - Fun          ->  Delta=+1.41 si match
    - Attractivité  ->  Delta=+1.34 si match
    - Intérêts réels  ->  Delta=+1.37 (noté EN FACE, pas déclaré)
    - Like ≥ 8     ->  35% de match (2.1x la base)

 Recommandation #1 : Pondérer Fun & Attractivité dans l'algo.
 Recommandation #2 : Créer un 'like score' composite.
 Recommandation #3 : Simplifier l'onboarding — la race et les intérêts déclarés n'ont pas de valeur prédictive."

